In [2]:
import torch
from torch.distributions import Categorical
from tqdm import trange

@torch.no_grad()
def test_rew_end_model_on_dataset(model, dataset, device, num_samples=100):
    """
    Check whether the reward-end model ever predicts non-zero reward.
    Only tests transitions with true non-zero reward.
    """
    model.eval()

    num_nonzero = 0
    num_high_confidence = 0
    total_checked = 0
    printed_examples = 0

    for _ in trange(num_samples):
        # Sample until we get a segment with a non-zero reward
        for _ in range(50):
            segment = dataset.sample_segment(seq_length=2)
            true_rew = segment.rew[-1].item()
            if true_rew != 0:
                break
        else:
            print("Couldn't find non-zero reward in 50 tries, skipping.")
            continue

        obs = segment.obs[:-1].unsqueeze(0).to(device)       # shape: (1, 1, C, H, W)
        act = segment.act[-1:].unsqueeze(0).to(device)        # shape: (1, 1)
        next_obs = segment.obs[1:].unsqueeze(0).to(device)    # shape: (1, 1, C, H, W)

        logits, *_ = model(obs, act, next_obs)
        probs = torch.softmax(logits, dim=-1).squeeze(0).squeeze(0).cpu()  # shape: (3,)

        sampled_rew = Categorical(logits=logits).sample().squeeze(1) - 1.0
        sampled_rew = sampled_rew.item()

        total_checked += 1
        if sampled_rew != 0:
            num_nonzero += 1

        if probs.max().item() > 0.9:
            num_high_confidence += 1

        if printed_examples < 5:
            print(f"\n--- Example {printed_examples+1} ---")
            print(f"True reward       : {true_rew}")
            print(f"Predicted probs   : {probs.tolist()}")
            print(f"Sampled reward    : {sampled_rew}")
            printed_examples += 1

    print("\n======= SUMMARY =======")
    print(f"Total checked transitions     : {total_checked}")
    print(f"Sampled non-zero rewards      : {num_nonzero}")
    print(f"High-confidence predictions   : {num_high_confidence}")



# === USAGE EXAMPLE ===

if __name__ == "__main__":
    from trainer import Trainer  # or wherever your Trainer class is defined
    import hydra
    from pathlib import Path
    from omegaconf import OmegaConf

    # Load config and init Trainer
    cfg = OmegaConf.load("config/trainer.yaml")
    trainer = Trainer(cfg=cfg, root_dir=Path("."))

    # Run the test
    test_rew_end_model_on_dataset(
        model=trainer.agent.rew_end_model,
        dataset=trainer.train_dataset,
        device=trainer._device,
        num_samples=100,
    )

FileNotFoundError: [Errno 2] No such file or directory: '/homes/53/fpinto/diamond/src/config/trainer.yaml'